In [ ]:
# Invoice Data Extraction Pipeline - Main Notebook

## Setup and Imports

import sys
import os
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from datetime import datetime, date
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import our modules
from src.pipeline import InvoiceExtractionPipeline
from src.google_drive_loader import GoogleDriveLoader
from src.models import DatabaseManager

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

## Step 1: Download Invoices from Google Drive (Optional)

# If you want to download from Google Drive, uncomment and run this section

"""
# Initialize Google Drive loader
gdrive = GoogleDriveLoader(credentials_path="../credentials.json")

# Option 1: Use folder ID directly
folder_id = os.getenv("GOOGLE_DRIVE_FOLDER_ID")

# Option 2: Extract from URL
# folder_url = "https://drive.google.com/drive/folders/YOUR_FOLDER_ID"
# folder_id = gdrive.get_folder_id_from_url(folder_url)

# Download all invoice files
downloaded_files = gdrive.download_folder(
    folder_id=folder_id,
    output_dir="../data/raw"
)

print(f"Downloaded {len(downloaded_files)} files")
"""

## Step 2: Initialize Pipeline

# Initialize the pipeline with configuration
pipeline = InvoiceExtractionPipeline(config_path="../config.yaml")

print("✓ Pipeline initialized")

## Step 3: Process Single File (Test)

# Test on a single file first
test_file = "../data/raw/sample_invoice.pdf"  # Update with your file

if os.path.exists(test_file):
    print(f"Processing test file: {test_file}")
    
    # Extract data
    data = pipeline.process_file(test_file, strategy="auto")
    
    # Display results
    if data:
        print("\n--- Extracted Data ---")
        for key, value in data.items():
            if key != 'line_items' and key != '_raw_text':
                print(f"{key}: {value}")
        
        if 'line_items' in data and data['line_items']:
            print(f"\nLine Items: {len(data['line_items'])} items")
            for i, item in enumerate(data['line_items'][:3], 1):
                print(f"  {i}. {item.get('description', 'N/A')}: ${item.get('line_total', 0):.2f}")
    else:
        print("Failed to extract data from test file")
else:
    print(f"Test file not found: {test_file}")
    print("Please place a sample invoice in data/raw/ or update the path")

## Step 4: Process All Invoices in Directory

# Process all invoices
invoice_dir = "../data/raw"

print(f"\nProcessing all invoices in {invoice_dir}...")

results = pipeline.process_directory(
    directory=invoice_dir,
    strategy="auto",
    parallel=True  # Use parallel processing
)

print(f"\n✓ Processed {len(results)} invoices successfully")

## Step 5: View Extracted Data

# Convert to DataFrame for easy viewing
if results:
    df = pd.DataFrame(results)
    
    # Remove complex columns for display
    display_df = df.drop(columns=['line_items', '_raw_text'], errors='ignore')
    
    print("\n--- First 5 Invoices ---")
    print(display_df.head())
    
    # Summary statistics
    print("\n--- Summary Statistics ---")
    print(f"Total Invoices: {len(df)}")
    print(f"Unique Vendors: {df['vendor_name'].nunique()}")
    print(f"Total Amount: ${df['total_amount'].sum():,.2f}")
    print(f"Date Range: {df['invoice_date'].min()} to {df['invoice_date'].max()}")
    print(f"Average Confidence: {df['confidence_score'].mean():.2%}")

## Step 6: Query Database

# Get all invoices from database
all_invoices_df = pipeline.export_all_data()

print("\n--- Database Contents ---")
print(f"Total invoices in database: {len(all_invoices_df)}")

# Query specific vendor
vendor_name = "Acme Corp"  # Update with actual vendor name
if vendor_name in all_invoices_df['vendor_name'].values:
    vendor_invoices = pipeline.query_invoices(vendor=vendor_name)
    print(f"\nInvoices from {vendor_name}: {len(vendor_invoices)}")
    print(vendor_invoices[['invoice_number', 'invoice_date', 'total_amount']])

# Get spending by vendor
spending_df = pipeline.get_spending_by_vendor()
print("\n--- Total Spend by Vendor ---")
print(spending_df.sort_values('total_spend', ascending=False).head(10))

## Step 7: Visualize Data

if len(all_invoices_df) > 0:
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Spending by Vendor (Top 10)
    spending_df = pipeline.get_spending_by_vendor()
    top_vendors = spending_df.nlargest(10, 'total_spend')
    axes[0, 0].barh(top_vendors['vendor_name'], top_vendors['total_spend'])
    axes[0, 0].set_xlabel('Total Spend ($)')
    axes[0, 0].set_title('Top 10 Vendors by Spend')
    axes[0, 0].invert_yaxis()
    
    # 2. Invoices Over Time
    all_invoices_df['invoice_date'] = pd.to_datetime(all_invoices_df['invoice_date'])
    monthly_counts = all_invoices_df.groupby(all_invoices_df['invoice_date'].dt.to_period('M')).size()
    axes[0, 1].plot(monthly_counts.index.astype(str), monthly_counts.values, marker='o')
    axes[0, 1].set_xlabel('Month')
    axes[0, 1].set_ylabel('Number of Invoices')
    axes[0, 1].set_title('Invoices Over Time')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Amount Distribution
    axes[1, 0].hist(all_invoices_df['total_amount'], bins=20, edgecolor='black')
    axes[1, 0].set_xlabel('Invoice Amount ($)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Invoice Amounts')
    
    # 4. Confidence Scores
    if 'confidence_score' in all_invoices_df.columns:
        axes[1, 1].hist(all_invoices_df['confidence_score'], bins=20, edgecolor='black')
        axes[1, 1].set_xlabel('Confidence Score')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Extraction Confidence Scores')
    
    plt.tight_layout()
    plt.savefig('../outputs/analysis_dashboard.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualizations saved to outputs/analysis_dashboard.png")

## Step 8: Advanced Queries

# Query invoices by date range
start_date = date(2024, 1, 1)
end_date = date(2024, 12, 31)

date_filtered = pipeline.query_invoices(start_date=start_date, end_date=end_date)
print(f"\nInvoices from {start_date} to {end_date}: {len(date_filtered)}")

# Get pipeline statistics
stats = pipeline.get_statistics()
print("\n--- Pipeline Statistics ---")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key}: ${value:,.2f}")
    else:
        print(f"{key}: {value}")

## Step 9: Export Results

# Export to CSV
output_csv = "../outputs/all_invoices.csv"
all_invoices_df.to_csv(output_csv, index=False)
print(f"\n✓ Exported all data to {output_csv}")

# Export spending summary
spending_output = "../outputs/vendor_spending.csv"
spending_df.to_csv(spending_output, index=False)
print(f"✓ Exported spending summary to {spending_output}")

## Summary

print("\n" + "="*50)
print("EXTRACTION PIPELINE COMPLETE")
print("="*50)
print(f"✓ Processed {len(results)} new invoices")
print(f"✓ Total invoices in database: {stats['total_invoices']}")
print(f"✓ Total amount: ${stats['total_amount']:,.2f}")
print(f"✓ Unique vendors: {stats['unique_vendors']}")
print(f"✓ Outputs saved to: outputs/")
print("="*50)